### In Class April 28

In [2]:
# importing libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score



### Importing, inspecting & splitting data

In [3]:
# load breast cancer data, inspect it, and split it
data=load_breast_cancer()
cancer_data=pd.DataFrame(data.data,columns=data.feature_names)
cancer_data["target"]=data.target
cancer_data["target_name"]=cancer_data["target"].map({0:"malignant",1:"benign"})
print(cancer_data.head())
print()
print(cancer_data["target_name"].value_counts())
print()
print("missing values:",cancer_data.isna().sum().sum())
X_cancer=cancer_data.drop(["target","target_name"],axis=1)
y_cancer=cancer_data["target"]
X_train_cancer,X_test_cancer,y_train_cancer,y_test_cancer=train_test_split(X_cancer,y_cancer,test_size=0.20,stratify=y_cancer,random_state=42)

   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst perimeter  worst area  worst smoothness  \
0          

### Defining functions

In [4]:
# helper function for cross validation comparison
def compare_models_cv(models,X_train,y_train,cv):
    cv_results=[]
    for name,model in models.items():
        scores=cross_validate(model,X_train,y_train,cv=cv,scoring=["accuracy","precision","recall","f1","balanced_accuracy"])
        cv_results.append({
            "model":name,
            "cv_accuracy":scores["test_accuracy"].mean(),
            "cv_precision":scores["test_precision"].mean(),
            "cv_recall":scores["test_recall"].mean(),
            "cv_f1":scores["test_f1"].mean(),
            "cv_balanced_accuracy":scores["test_balanced_accuracy"].mean()
        })
    return pd.DataFrame(cv_results).sort_values("cv_balanced_accuracy",ascending=False)

In [5]:
# helper function for threshold tuning
def evaluate_thresholds(y_true,probs,thresholds=None):
    if thresholds is None:
        thresholds=np.linspace(0.1,0.9,17) # change 17 to higher number to test more thresholds
    rows=[]
    for t in thresholds:
        preds=(probs[:,1]>=t).astype(int)
        rows.append({
            "threshold":t,
            "precision":precision_score(y_true,preds,zero_division=0),
            "recall":recall_score(y_true,preds,zero_division=0),
            "f1":f1_score(y_true,preds,zero_division=0),
            "balanced_accuracy":balanced_accuracy_score(y_true,preds)
        })
    return pd.DataFrame(rows)

In [6]:
# helper function for held out test evaluation
def evaluate_models_test(models,X_train,X_test,y_train,y_test):
    trained_models={}
    test_results=[]
    for name,model in models.items():
        model.fit(X_train,y_train)
        trained_models[name]=model
        preds=model.predict(X_test)
        test_results.append({
            "model":name,
            "test_accuracy":accuracy_score(y_test,preds),
            "test_precision":precision_score(y_test,preds,zero_division=0),
            "test_recall":recall_score(y_test,preds,zero_division=0),
            "test_f1":f1_score(y_test,preds,zero_division=0),
            "test_balanced_accuracy":balanced_accuracy_score(y_test,preds)
        })
    return trained_models,pd.DataFrame(test_results).sort_values("test_balanced_accuracy",ascending=False)

### Defining models & CV

In [7]:
# define cross validation and cancer models
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
models_cancer={
    "Naive Bayes":GaussianNB(),
    "Logistic Regression":Pipeline([("scaler",StandardScaler()),("model",LogisticRegression(max_iter=2000))]),
    "Random Forest":RandomForestClassifier(random_state=42)
}

### Model comparison with cross validation

In [8]:
# compare cancer models with cross validation
compare_models_cv(models_cancer,X_train_cancer,y_train_cancer,cv)

,model,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_balanced_accuracy
1,Logistic Regression,0.978022,0.979472,0.985965,0.982544,0.975335
2,Random Forest,0.962637,0.975519,0.964912,0.969935,0.961868
0,Naive Bayes,0.940659,0.942185,0.964912,0.953280,0.932456


### Baseline model evaluation on test data

In [9]:
# fit models on full training data and evaluate on held-out test set
trained_models_cancer,test_results_cancer=evaluate_models_test(
    models_cancer,X_train_cancer,X_test_cancer,y_train_cancer,y_test_cancer
)
test_results_cancer

,model,test_accuracy,test_precision,test_recall,test_f1,test_balanced_accuracy
1,Logistic Regression,0.982456,0.986111,0.986111,0.986111,0.981151
2,Random Forest,0.956140,0.958904,0.972222,0.965517,0.950397
0,Naive Bayes,0.938596,0.945205,0.958333,0.951724,0.931548


### Threshold tuning

In [10]:
# threshold tuning on logistic regression
model=trained_models_cancer["Logistic Regression"]
probs_cancer_lr=model.predict_proba(X_test_cancer)

threshold_results_cancer_lr=evaluate_thresholds(y_test_cancer,probs_cancer_lr)
best=threshold_results_cancer_lr.sort_values("f1",ascending=False).iloc[0]

default_preds=(probs_cancer_lr[:,1]>=0.5).astype(int)
print(threshold_results_cancer_lr.sort_values("f1",ascending=False).head())
print()
print("default threshold:",0.5)
print("default f1:",round(f1_score(y_test_cancer,default_preds),4))
print("default balanced accuracy:",round(balanced_accuracy_score(y_test_cancer,default_preds),4))
print("best threshold (for f1):",round(best["threshold"],4))
print("best f1:",round(best["f1"],4))
print("balanced accuracy at best-f1 threshold:",round(best["balanced_accuracy"],4))

   threshold  precision    recall        f1  balanced_accuracy
2       0.20   0.972973  1.000000  0.986301           0.976190
3       0.25   0.972973  1.000000  0.986301           0.976190
4       0.30   0.972973  1.000000  0.986301           0.976190
5       0.35   0.972973  1.000000  0.986301           0.976190
8       0.50   0.986111  0.986111  0.986111           0.981151

default threshold: 0.5
default f1: 0.9861
default balanced accuracy: 0.9812
best threshold (for f1): 0.2
best f1: 0.9863
balanced accuracy at best-f1 threshold: 0.9762


In [13]:
# threshold tuning on naive bayes
model=trained_models_cancer["Naive Bayes"]
probs_cancer_nb=model.predict_proba(X_test_cancer)

threshold_results_cancer_nb=evaluate_thresholds(y_test_cancer,probs_cancer_nb)
best=threshold_results_cancer_nb.sort_values("f1",ascending=False).iloc[0]

default_preds=(probs_cancer_nb[:,1]>=0.5).astype(int)
print(threshold_results_cancer_nb.sort_values("f1",ascending=False).head())
print()
print("default threshold:",0.5)
print("default f1:",round(f1_score(y_test_cancer,default_preds),4))
print("default balanced accuracy:",round(balanced_accuracy_score(y_test_cancer,default_preds),4))
print("best threshold (for f1):",round(best["threshold"],4))
print("best f1:",round(best["f1"],4))
print("balanced accuracy at best-f1 threshold:",round(best["balanced_accuracy"],4))

   threshold  precision    recall        f1  balanced_accuracy
0       0.10   0.946667  0.986111  0.965986           0.945437
2       0.20   0.946667  0.986111  0.965986           0.945437
3       0.25   0.946667  0.986111  0.965986           0.945437
1       0.15   0.946667  0.986111  0.965986           0.945437
4       0.30   0.945946  0.972222  0.958904           0.938492

default threshold: 0.5
default f1: 0.9517
default balanced accuracy: 0.9315
best threshold (for f1): 0.1
best f1: 0.966
balanced accuracy at best-f1 threshold: 0.9454


In [14]:
# threshold tuning on random forest
model=trained_models_cancer["Random Forest"]
probs_cancer_rf=model.predict_proba(X_test_cancer)

threshold_results_cancer_rf=evaluate_thresholds(y_test_cancer,probs_cancer_rf)
best=threshold_results_cancer_rf.sort_values("f1",ascending=False).iloc[0]

default_preds=(probs_cancer_rf[:,1]>=0.5).astype(int)
print(threshold_results_cancer_rf.sort_values("f1",ascending=False).head())
print()
print("default threshold:",0.5)
print("default f1:",round(f1_score(y_test_cancer,default_preds),4))
print("default balanced accuracy:",round(balanced_accuracy_score(y_test_cancer,default_preds),4))
print("best threshold (for f1):",round(best["threshold"],4))
print("best f1:",round(best["f1"],4))
print("balanced accuracy at best-f1 threshold:",round(best["balanced_accuracy"],4))

   threshold  precision    recall        f1  balanced_accuracy
2       0.20   0.935065  1.000000  0.966443           0.940476
3       0.25   0.935065  1.000000  0.966443           0.940476
1       0.15   0.935065  1.000000  0.966443           0.940476
8       0.50   0.958904  0.972222  0.965517           0.950397
6       0.40   0.958904  0.972222  0.965517           0.950397

default threshold: 0.5
default f1: 0.9655
default balanced accuracy: 0.9504
best threshold (for f1): 0.2
best f1: 0.9664
balanced accuracy at best-f1 threshold: 0.9405


### Deploying optimal thresholds with models

In [15]:
# deploy predictions with explicit threshold

# logistic regression
preds_lr = (trained_models_cancer["Logistic Regression"].predict_proba(X_test_cancer)[:,1] >= 0.20).astype(int)
print("Logistic Regression")
print("precision:", round(precision_score(y_test_cancer, preds_lr, zero_division=0),4))
print("recall:", round(recall_score(y_test_cancer, preds_lr, zero_division=0),4))
print("f1:", round(f1_score(y_test_cancer, preds_lr, zero_division=0),4))
print("balanced accuracy:", round(balanced_accuracy_score(y_test_cancer, preds_lr),4))
print()

# naive bayes
preds_nb = (trained_models_cancer["Naive Bayes"].predict_proba(X_test_cancer)[:,1] >= 0.20).astype(int)
print("Naive Bayes")
print("precision:", round(precision_score(y_test_cancer, preds_nb, zero_division=0),4))
print("recall:", round(recall_score(y_test_cancer, preds_nb, zero_division=0),4))
print("f1:", round(f1_score(y_test_cancer, preds_nb, zero_division=0),4))
print("balanced accuracy:", round(balanced_accuracy_score(y_test_cancer, preds_nb),4))
print()

# random forest
preds_rf = (trained_models_cancer["Random Forest"].predict_proba(X_test_cancer)[:,1] >= 0.20).astype(int)
print("Random Forest")
print("precision:", round(precision_score(y_test_cancer, preds_rf, zero_division=0),4))
print("recall:", round(recall_score(y_test_cancer, preds_rf, zero_division=0),4))
print("f1:", round(f1_score(y_test_cancer, preds_rf, zero_division=0),4))
print("balanced accuracy:", round(balanced_accuracy_score(y_test_cancer, preds_rf),4))

Logistic Regression
precision: 0.973
recall: 1.0
f1: 0.9863
balanced accuracy: 0.9762

Naive Bayes
precision: 0.9467
recall: 0.9861
f1: 0.966
balanced accuracy: 0.9454

Random Forest
precision: 0.9351
recall: 1.0
f1: 0.9664
balanced accuracy: 0.9405
